# Notebook: `03. Análises de Schema`
---
## Descrição

Esse notebook faz uma análise preliminar do Schema dos arquivos `.parquet` dos datasets da Taxi and Limousine Comission (TLC). Essa etapa foi aplicada pois em versões anteriores do pipeline houve erros ao ingerir os dados brutos para as tabelas na camada `bronze`

Esses erros tiveram como causa raiz a impossibilidade de utilizar a opção `mergeSchema` no Databricks free edition, e pela decorrência das seguintes divergências de schemas entre os arquivos da TLC:

* Criação de novos campos no decorrer do tempo, como `dropoff_datetime` e `DOLocationID` para a base `forhirevehicles`;
* Alteração constante de tipagem de dados em campos já existentes, como alternância entre os tipos `long`, `int` e `double` entre meses ou entre anos;
* Alteração de grafia de campos já existentes, e persistência do campo com a grafia antiga, porém vazio, dentro de um mesmo arquivo `.parquet`, observado na base `yellowtaxi` (`Airport_fee` vs `airport_fee`)

Com o entendimento da divergência de tipagens de dados no histórico das bases, foi possível definir um `StructType` na camada `bronze` com as tipagens mais permissivas para não gerar perda de dados.

As colunas e tipos de dados por tabela foram descritas ao final do notebook.

Importação de bibliotecas usadas.

In [0]:
from collections import defaultdict

**leituraParquet**: Função de diagnóstico de schema. Lê o schema de cada arquivo `.parquet` presente no caminho informado e, para cada coluna, compara os tipos de dado encontrados ao longo dos arquivos. Ao final, imprime um resumo indicando quais colunas possuem tipo consistente em todos os arquivos e quais apresentam divergência, detalhando os tipos distintos encontrados, a quantidade de arquivos de cada tipo e exemplos (primeiro e último arquivo).

In [0]:
def leituraParquet(path):
    files = dbutils.fs.ls(path)
    parquet_files = [f.path for f in files if f.name.endswith(".parquet")]

    schemas = {}
    for file_path in parquet_files:
        df = spark.read.parquet(file_path)
        schemas[file_path.split("/")[-1]] = df.schema

    column_types = defaultdict(dict)

    for fname, schema in schemas.items():
        for field in schema.fields:
            column_types[field.name][fname] = str(field.dataType)

    print("=" * 80)
    print(f'{"SUMMARY: Distinct Types per Column for "}{path}')
    print("=" * 80)

    for col in sorted(column_types.keys()):
        types_by_file = column_types[col]
        unique_types = sorted(set(types_by_file.values()))
        
        if len(unique_types) == 1:
            print(f"\n{col}: {unique_types[0]} (consistent across all files)")
        else:
            print(f"\n{col}: {len(unique_types)} different types found")
            for dtype in unique_types:

                files_with_type = [f for f, t in types_by_file.items() if t == dtype]
                print(f"  • {dtype}: {len(files_with_type)} files")

                if len(files_with_type) > 2:
                    print(f"    Examples: {files_with_type[0]}, ..., {files_with_type[-1]}")
                else:
                    for f in files_with_type:
                        print(f"    {f}")

Invoca a função `leituraParquet` a partir dos caminhos dos volumes com os arquivos brutos de cada dataset da TLC.

In [0]:
path = [
    "/Volumes/mvp/staging/yellow_taxi",
    "/Volumes/mvp/staging/green_taxi",
    "/Volumes/mvp/staging/fhv",
    "/Volumes/mvp/staging/fhvhv"
]

for p in path:
    leituraParquet(p)

SUMMARY: Distinct Types per Column for /Volumes/mvp/staging/yellow_taxi

Airport_fee: DoubleType() (consistent across all files)

DOLocationID: 2 different types found
  • IntegerType(): 40 files
    Examples: yellow_tripdata_2023-02.parquet, ..., yellow_tripdata_2026-05.parquet
  • LongType(): 85 files
    Examples: yellow_tripdata_2016-01.parquet, ..., yellow_tripdata_2023-01.parquet

PULocationID: 2 different types found
  • IntegerType(): 40 files
    Examples: yellow_tripdata_2023-02.parquet, ..., yellow_tripdata_2026-05.parquet
  • LongType(): 85 files
    Examples: yellow_tripdata_2016-01.parquet, ..., yellow_tripdata_2023-01.parquet

RatecodeID: 2 different types found
  • DoubleType(): 55 files
    Examples: yellow_tripdata_2018-07.parquet, ..., yellow_tripdata_2023-01.parquet
  • LongType(): 70 files
    Examples: yellow_tripdata_2016-01.parquet, ..., yellow_tripdata_2026-05.parquet

VendorID: 2 different types found
  • IntegerType(): 40 files
    Examples: yellow_tripdata_2

A análise acima indica as seguintes configurações de campos:

###Yellow Taxi:
| campo | Tipo | Observação |
|--------------|------|------------|
| Airport_fee | DoubleType | Variante de `airport_fee`.<br>Há apenas uma ocorrência |
| DOLocationID | IntegerType<br>LongType |            |
| PULocationID | IntegerType<br>LongType |  |
| RatecodeID | DoubleType<br>LongType |  |
| VendorID | IntegerType<br>LongType |  |
| airport_fee | DoubleType<br>IntegerType |  |
| cbd_congestion_fee | DoubleType |  |
| congestion_surcharge | DoubleType<br>IntegerType |  |
| extra | DoubleType |  |
| fare_amount | DoubleType |  |
| improvement_surcharge | DoubleType |  |
| mta_tax | DoubleType |  |
| passenger_count | DoubleType<br>LongType |  |
| payment_type | LongType |  |
| store_and_fwd_flag | StringType |  |
| tip_amount | DoubleType |  |
| tolls_amount | DoubleType |  |
| total_amount | DoubleType |  |
| tpep_dropoff_datetime | TimestampNTZType |  |
| tpep_pickup_datetime | TimestampNTZType |  |
| trip_distance | DoubleType |  |

###Green Taxi
| campo | Tipo | Observação |
|--------------|------|------------|
| DOLocationID | IntegerType<br> LongType |  |
| PULocationID | IntegerType<br>LongType |  |
| RatecodeID | DoubleType<br>LongType |  |
| VendorID | IntegerType<br>LongType |  |
| cbd_congestion_fee | DoubleType |  |
| congestion_surcharge | DoubleType<br>IntegerType |  |
| ehail_fee | DoubleType<br>IntegerType |  |
| extra | DoubleType |  |
| fare_amount | DoubleType |  |
| improvement_surcharge | DoubleType |  |
| lpep_dropoff_datetime | TimestampNTZType |  |
| lpep_pickup_datetime | TimestampNTZType |  |
| mta_tax | DoubleType |  |
| passenger_count | DoubleType<br>LongType |  |
| payment_type | DoubleType<br>LongType |  |
| store_and_fwd_flag | StringType |  |
| tip_amount | DoubleType |  |
| tolls_amount | tolls_amount |  |
| total_amount | DoubleType |  |
| trip_distance | DoubleType |  |
| trip_type | DoubleType<br>LongType |  |

###For Hire Vehicles (FHV)
| campo | Tipo | Observação |
|--------------|------|------------|
| Affiliated_base_number | StringType |  |
| DOlocationID | DoubleType<br>LongType |  |
| PUlocationID | DoubleType<br>LongType |  |
| SR_Flag | DoubleType<br>IntegerType<br>LongType |  |
| dispatching_base_num | StringType |  |
| dropOff_datetime | TimestampNTZType |  |
| pickup_datetime | TimestampNTZType |  |

###For Hire Vehicle High Volume (FHVHV)

| campo | Tipo | Observação |
|--------------|------|------------|
| DOLocationID | IntegerType<br>LongType |  |
| PULocationID | IntegerType<br>LongType |  |
| access_a_ride_flag | StringType |  |
| airport_fee | DoubleType<br>IntegerType |  |
| base_passenger_fare | DoubleType |  |
| bcf | DoubleType |  |
| cbd_congestion_fee | DoubleType |  |
| congestion_surcharge | DoubleType |  |
| dispatching_base_num | StringType |  |
| driver_pay | DoubleType |  |
| dropoff_datetime | TimestampNTZType |  |
| hvfhs_license_num | StringType |  |
| on_scene_datetime | TimestampNTZType |  |
| originating_base_num | StringType |  |
| pickup_datetime | TimestampNTZType |  |
| request_datetime | TimestampNTZType |  |
| sales_tax | DoubleType |  |
| shared_match_flag | StringType |  |
| shared_request_flag | StringType |  |
| tips | DoubleType |  |
| tolls | DoubleType |  |
| trip_miles | DoubleType |  |
| trip_time | LongType |  |
| wav_match_flag | IntegerType<br>StringType |  |
| wav_request_flag | StringType |  |